In [ ]:
#| default_exp project

In [ ]:
#| export
from __future__ import annotations
from dataclasses import dataclass, field
from fastcore.all import Path

In [ ]:
#| export
@dataclass
class Step:
    "One command in a pipeline."
    id: str
    label: str
    cmd: str
    doc: str = ''
    needs: list = field(default_factory=list)   # environment keys the command reads
    argv: list = field(default_factory=list)    # spelled out, where `cmd` would not split correctly
    meta: dict = field(default_factory=dict)
    def dict(self): return {'id': self.id, 'label': self.label, 'cmd': self.cmd, 'doc': self.doc,
                            'needs': list(self.needs), 'meta': dict(self.meta)}

In [ ]:
#| export
def _toml(root, name='pyproject.toml'):
    "One TOML file as a dict, or `{}` where it is missing or unreadable."
    import tomllib
    try:
        with open(Path(root)/name, 'rb') as f: return tomllib.load(f)
    except (OSError, ValueError): return {}

def nbdev_project(root):
    "An nbdev project: `settings.ini` for nbdev 2, or `[tool.nbdev]` in `pyproject.toml`."
    try:
        text = (Path(root)/'settings.ini').read_text(encoding='utf-8', errors='replace')
        if '[DEFAULT]' in text and any(k in text for k in ('nbs_path', 'lib_name', 'doc_path')):
            return True
    except OSError: pass
    data = _toml(root)
    return 'nbdev' in (data.get('tool') or {}) or 'nbdev' in ((data.get('project') or {}).get('entry-points') or {})

def rust_project(root):
    "A crate: something cargo can build, whether or not Python ever sees it."
    return (Path(root)/'Cargo.toml').exists()

def maturin_project(root):
    "A crate that is also a Python package: maturin builds the wheel, so the release is PyPI's."
    if not rust_project(root): return False
    data = _toml(root)
    backend = str((data.get('build-system') or {}).get('build-backend') or '')
    return backend.startswith('maturin') or 'maturin' in (data.get('tool') or {})

def fastship_project(root):
    "A Python package fastship releases: a `pyproject.toml`, and not an nbdev project."
    return (Path(root)/'pyproject.toml').exists() and not nbdev_project(root)

def app_project(root):
    "Whether the project packages itself into a desktop app, by the `setup_app.py` convention."
    root = Path(root)
    return (root/'tools'/'build_release.py').exists() and (root/'setup_app.py').exists()

In [ ]:
#| export
NBDEV_STEPS = [
    Step('prepare', 'nbdev_prepare', 'nbdev_prepare',
        'Export the notebooks, run the tests, clean the metadata, rebuild the README.'),
    Step('bump', 'bump version', 'nbdev_bump_version',
        'Raise the version in settings.ini and commit it.'),
    Step('gh', 'GitHub release', 'nbdev_release_gh',
        'Tag the version and create the GitHub release from CHANGELOG.', ['GITHUB_TOKEN']),
    Step('pypi', 'PyPI release', 'nbdev_pypi',
        'Build the wheel and upload it to PyPI.', ['TWINE_USERNAME', 'TWINE_PASSWORD']),
]

FASTSHIP_STEPS = [
    Step('test', 'test', 'python -m pytest -q', 'Run the test suite before anything is published.'),
    Step('changelog', 'changelog', 'ship-changelog',
        'Rewrite CHANGELOG.md from the issues closed since the last release.', ['GITHUB_TOKEN']),
    Step('gh', 'GitHub release', 'ship-gh --no_changelog --no_editor --yes',
        'Commit, push, tag, and create the GitHub release. The changelog step already wrote the notes.',
        ['GITHUB_TOKEN']),
    Step('pypi', 'PyPI release', 'ship-pypi',
        'Build the sdist and wheel, twine check them, then upload.',
        ['TWINE_USERNAME', 'TWINE_PASSWORD']),
    Step('bump', 'bump version', 'ship-bump',
        'Raise the version for the next cycle. fastship bumps after releasing, not before.'),
]

PLAIN_STEPS = [
    Step('test', 'test', 'python -m pytest -q', 'Run the test suite before anything is published.'),
    Step('build', 'build', 'python -m build', 'Build the sdist and wheel into dist/.'),
    Step('gh', 'GitHub release', 'gh release create --generate-notes',
        'Tag and publish a GitHub release.', ['GITHUB_TOKEN']),
    Step('pypi', 'PyPI release', 'python -m twine upload dist/*',
        'Upload the built distributions to PyPI.', ['TWINE_USERNAME', 'TWINE_PASSWORD']),
]

MATURIN_STEPS = [
    Step('test', 'test', 'cargo test', "Run the crate's own tests before anything is published."),
    Step('wheel', 'build the wheel', 'maturin build --release --out dist',
        'Build here first. A tag that fails to compile on the runners is a tag you have to undo.'),
    Step('ci', 'refresh the workflow', 'maturin generate-ci github -o .github/workflows/CI.yml',
        'maturin writes the matrix: linux, musllinux, macOS, Windows, and the sdist. Rewriting it '
        'from maturin keeps it right when your Python range or targets change.'),
    Step('changelog', 'changelog', 'ship-changelog',
        'Rewrite CHANGELOG.md from the issues closed since the last release.', ['GITHUB_TOKEN']),
    Step('release', 'tag and push', 'ship-release',
        'Tag the version from Cargo.toml and push it. The workflow builds every platform\'s wheel '
        'and publishes them, so nothing is uploaded from here.', ['GITHUB_TOKEN']),
]

CRATE_STEPS = [
    Step('test', 'test', 'cargo test', "Run the crate's tests before anything is published."),
    Step('clippy', 'clippy', 'cargo clippy --all-targets -- -D warnings',
        'The lints. A crate other people will read is worth the stricter pass.'),
    Step('package', 'package', 'cargo package',
        'Build the .crate the registry would receive, and verify it on its own.'),
    Step('publish', 'publish to crates.io', 'cargo publish',
        'Upload it. A version on crates.io cannot be replaced, only yanked.',
        ['CARGO_REGISTRY_TOKEN']),
]

APP_STEPS = [
    Step('app', 'desktop app', 'python tools/build_release.py',
        'Build the platform bundle: a .app and a DMG on macOS, a zipped folder on Windows.'),
    Step('assets', 'attach the app', 'python tools/build_release.py upload',
        'Upload what dist/ holds to the release this version just tagged.', ['GITHUB_TOKEN']),
]

#: Every flow, by the name `release_flow` answers with. Add one and `default_steps` can reach it.
FLOWS = {'nbdev': NBDEV_STEPS, 'fastship': FASTSHIP_STEPS, 'plain': PLAIN_STEPS,
         'maturin': MATURIN_STEPS, 'crate': CRATE_STEPS}

In [ ]:
#| export
def release_flow(root):
    """Which release flow this project uses.

    Rust is asked before fastship, because a maturin crate has a `pyproject.toml` too and the
    fastship flow would try to `ship-pypi` a wheel that only the runners can build.
    """
    if nbdev_project(root): return 'nbdev'
    if maturin_project(root): return 'maturin'
    if rust_project(root): return 'crate'
    return 'fastship' if fastship_project(root) else 'plain'

def _with_app(steps):
    "The app bundle and its upload, after the last step that publishes and before any that bumps."
    at = [i for i, s in enumerate(steps) if s.id in ('gh', 'pypi')]
    i = max(at) + 1 if at else len(steps)
    return [*steps[:i], *APP_STEPS, *steps[i:]]

def default_steps(root, flows=None):
    "The steps this kind of project is released by, with the desktop bundle where there is one."
    steps = list((flows or FLOWS)[release_flow(root)])
    return _with_app(steps) if app_project(root) else steps

In [ ]:
#| export
class Project:
    "What kind of project one folder holds, and what releasing it takes."
    def __init__(self, root, flows=None):
        self.root, self.flows = Path(root).expanduser().resolve(), flows or FLOWS
    @property
    def kind(self): return release_flow(self.root)
    @property
    def packages_an_app(self): return app_project(self.root)
    def steps(self): return default_steps(self.root, self.flows)
    def dict(self):
        return {'root': str(self.root), 'kind': self.kind, 'app': self.packages_an_app,
                'steps': [s.dict() for s in self.steps()]}
    def __repr__(self): return f'Project({self.root.name}, {self.kind})'